# 讲课笔记：Day3 下午 — 企业知识助手 Capstone（3.5h）

> **配套 Notebook**: `Day3_下午_企业知识助手Capstone.ipynb`（共 51 个 Cell）

---

## 时间总览

| 时间段 | 内容 | Cell 范围 | 时长 |
|--------|------|-----------|------|
| 0:00-0:15 | 开场 + 架构介绍 + 环境准备 | Cell 0-4 | 15min |
| 0:15-0:30 | Step 1: 构建知识库 | Cell 5-8 | 15min |
| 0:30-0:45 | Checkpoint 1 + Step 2: RAG 回答 | Cell 9-12 | 15min |
| 0:45-1:00 | Checkpoint 2 + Step 3: Agent 路由 | Cell 13-17 | 15min |
| 1:00-1:10 | **休息** + 回顾 | Cell 18-20 | 10min |
| 1:10-1:30 | Checkpoint 3 + Step 4: 评测管道 | Cell 21-23 | 20min |
| 1:30-1:50 | Checkpoint 4 + Step 5: 多轮对话 | Cell 24-28 | 20min |
| 1:50-2:10 | Checkpoint 5 + Step 6: 自由扩展 | Cell 29-34 | 20min |
| 2:10-2:30 | Checkpoint 6: 综合演示 | Cell 35-36 | 20min |
| 2:30-2:50 | 生产化考量 | Cell 37-41 | 20min |
| 2:50-3:10 | 一周行动计划 | Cell 42-44 | 20min |
| 3:10-3:20 | 框架对比 | Cell 45-47 | 10min |
| 3:20-3:30 | 课程总结 + 寄语 | Cell 48-50 | 10min |

---

## 课前准备清单

- [ ] 确认 `.env` 正常（API Key + 模型配置）
- [ ] 确认前面 5 个 Notebook 已全部跑通
- [ ] 清空本 Notebook 所有输出
- [ ] 白板上画好系统架构图（Agent -> 4个工具 -> RAG -> Judge）
- [ ] 准备"星辰科技"场景故事背景

---

## 开场：系统架构介绍（0:00-0:15）

📍 **运行 Cell 0-4**

### Cell 0: 课程标题 + 系统架构

🗣 **讲课话术**:

> 好，大家下午好！今天下午是我们三天培训的**重头戏**——Capstone 项目。
>
> 前面五个 Notebook，我们分别学了训练循环、Tokenizer、Embedding、Attention、Transformer、预训练、SFT、LoRA、DPO、RAG、Agent......今天下午，**全部串起来**。
>
> 你们要亲手搭建一个能上生产的企业知识助手。
>
> 先看架构图（指白板）：

```
用户输入
  |
  v
ReAct Agent（LLM 驱动的调度员）
  |-- 工具1: rag_search（知识库检索）
  |-- 工具2: calculator（数学计算）
  |-- 工具3: code_executor（代码执行）
  +-- 工具4: direct_answer（通用回答）
  |
  v
RAG 增强回答 + 来源引用
  |
  v
LLM-as-Judge 自动评测
```

> Agent 是个"调度员"——用户问一个问题，它不是直接丢给 LLM，而是先想一想：这个问题该用哪个工具？去知识库搜？还是算数？还是跑代码？还是直接回答？
>
> 我们有 6 个 Checkpoint，像打游戏通关一样，一步步搭完整个系统。

### Cell 1: 场景设定

🗣 **讲课话术**:

> 我们的客户是**星辰科技有限公司**。你是他们的 AI 工程师，要给公司做一个内部知识助手。公司有三类文档：
> - **HR 制度**：请假、差旅报销、节假日
> - **产品手册**：StarLink（IoT 设备）、StarView（数据可视化平台）
> - **技术文档**：内部 API 规范、部署指南
>
> 员工可以用自然语言提问，系统自动检索文档并生成回答。

### Cell 3-4: 环境初始化

🗣 "快速跑一下环境初始化。应该看到 LLM 和 Embedding 都就绪。如果有报错，检查 .env 文件。"

👀 **确认输出**:
- `[LLM] dashscope / qwen-plus`
- `[Embedding] dashscope / text-embedding-v3`

❓ **常见问题**:
- Q: "API Key 过期了怎么办？" -> A: "去阿里云百炼控制台重新生成一个，改 .env 文件"


---

## Step 1: 构建知识库（0:15-0:30）

📍 **运行 Cell 5-8**

### Cell 6: 公司文档定义

⏱ 讲解 5 分钟

🗣 **讲课话术**:

> 看这个 `COMPANY_DOCUMENTS` 字典。6 个文档，3 个类别。
>
> **HR 类**：
> - 请假制度——工龄 1-10 年有 5 天年假，10-20 年 10 天，20 年以上 15 天。住宿报销上限 500 元/天
> - 差旅报销——飞机票经济舱、高铁二等座，出差补贴 100 元/天
>
> **产品类**：
> - StarLink——IoT 智能设备，有基础版 SL-3000（2999元）和 Pro 版 SL-3000 Pro（4999元，支持 5G）
> - StarView——数据可视化平台，支持 50+ 图表类型，可连 MySQL/PostgreSQL/MongoDB
>
> **技术类**：
> - API 规范——RESTful 风格，JWT 认证，P99 延迟要求 < 200ms
> - 部署指南——Docker + K8s，最低 4C8G，推荐 8C16G
>
> 这些内容都是我模拟的，实际企业里就是 Confluence 页面、PDF 手册、Word 文档。

### Cell 7: 文档分块

🗣 **讲课话术**:

> 文档太长不能直接塞给 LLM，要**分块**。
>
> 参数：`chunk_size=300`（每块最多 300 字），`overlap=50`（块之间重叠 50 字，保证上下文不断裂）。
>
> 6 个文档 -> 10 个块。为什么不是 6 个？因为长文档会被切成多块。

👀 **指出重点**:
- 打印的分块结果：每个块的标题、类别、字符数
- 总共 10 个块

❓ **常见问题**:
- Q: "chunk_size 怎么定？" -> A: "取决于 embedding 模型支持的最大长度和检索精度需求。300-500 是常用范围。太大了检索不精确，太小了丢失上下文"
- Q: "overlap 有什么用？" -> A: "防止一个概念被切成两半。比如一句话正好在边界上，overlap 让两个块都能看到这句话"

### Cell 8: 构建向量数据库

🗣 "用 SimpleVectorStore 把分块后的文档变成向量存起来。这一步实际就是调 embedding 模型，把文字变成 1024 维的数字。"

👀 **确认输出**:
- 10 个文档块全部嵌入成功
- 向量维度 1024

---

## Checkpoint 1: 知识库搜索（0:30-0:35）

📍 **运行 Cell 9-10**

### Cell 9: Checkpoint 说明

🗣 "第一个关卡！你们要实现一个 `search_knowledge` 函数。核心就是调 `vector_store.search()`，但要加一个**阈值过滤**——相似度太低的结果不要返回，防止返回不相关的内容。"

### Cell 10: 学员实现搜索函数

⏱ 给 **5 分钟** 动手时间

🗣 **给提示的节奏**:
- 0-2分钟：让学员自己试
- 2分钟：提示 "vector_store.search() 返回的每个结果有 score 字段"
- 4分钟：如果还没做出来，给出核心代码

```python
def search_knowledge(query, top_k=3, threshold=0.3):
    results = vector_store.search(query, top_k=top_k, threshold=threshold)
    return [r for r in results if r['score'] >= threshold]
```

👀 **测试验证**:
- 相关查询（"请假制度"）-> 返回结果，score > 0.5
- 无关查询（"今天天气怎么样"）-> 阈值过滤，返回空列表

🗣 "看到了吗？无关的问题被过滤掉了！这就是阈值的作用——防止系统硬编回答不知道的问题。"


---

## Step 2 + Checkpoint 2: RAG 回答系统（0:35-0:50）

📍 **运行 Cell 11-14**

### Cell 12: KnowledgeRAG 类

⏱ 讲解 5 分钟

🗣 **讲课话术**:

> 上一步我们能搜索了，现在要把搜到的结果喂给 LLM，让它生成完整的回答。
>
> 看这个 RAG 类的核心逻辑：
> 1. 用户问题 -> 向量检索 Top-K 文档片段
> 2. 把片段拼接成 context
> 3. Prompt 里写明："基于以下参考信息回答。如果参考信息不足，请说明。"
> 4. LLM 生成回答 + 标注来源
>
> **temperature=0.3**——为什么不用 0？因为 0 太死板，0.3 给一点点灵活性但不会乱说。

### Cell 14: Checkpoint 2 — 测试 RAG

⏱ 给 **5 分钟** 动手时间

🗣 "准备 7 个测试问题，覆盖 HR、产品、技术三个领域。注意要包含至少一个**超出知识库范围的问题**——系统应该诚实说'不知道'。"

👀 **指出重点**:
- HR 问题："工作5年有几天年假？" -> 回答 "5天"，引用来源 "hr_leave"
- 产品问题："StarLink Pro 多少钱？" -> 回答 "4999元"
- 越界问题："北京今天天气怎么样？" -> 回答 "抱歉，参考信息中未提及"

🗣 "**这就是 RAG 防幻觉的关键！** 模型知道自己不知道什么。上午讲 RAG 时说过——给模型一本参考书，它就只说书里有的。"

❓ **常见问题**:
- Q: "为什么有时候回答不够准确？" -> A: "检查分块质量。如果答案跨了两个块，可能检索只找到一半"
- Q: "top_k 设几？" -> A: "3-5 是常用范围。太多了会引入噪声，太少了可能漏掉关键信息"


---

## Step 3 + Checkpoint 3: Agent 路由（0:50-1:00）

📍 **运行 Cell 15-19**

### Cell 16: 工具定义

🗣 **讲课话术**:

> RAG 只会一件事——查文档。但员工的问题五花八门：
>
> | 问题类型 | 例子 | 需要的工具 |
> |---------|------|-----------|
> | 企业知识 | "请假制度是什么？" | rag_search |
> | 数学计算 | "5天住宿费总共多少？" | calculator |
> | 写代码 | "写个 Python 脚本处理 CSV" | code_executor |
> | 通用问答 | "什么是机器学习？" | direct_answer |
>
> 所以我们需要 **Agent**——一个能自动选工具的调度员。

### Cell 17: ReAct Agent

🗣 "看 KnowledgeAgent 的核心循环：Thought -> Action -> Observation。最多 5 步，防止死循环。"

### Cell 19: Checkpoint 3 — 测试路由

⏱ 给 **5 分钟** 动手时间

🗣 "准备 4 个测试用例，每个对应一个工具。跑完后注意看 Agent 打印的 `Action` 字段——它选对工具了吗？"

👀 **指出重点**:
- 知识查询 -> `Action: rag_search` ✓
- 数学计算 -> `Action: calculator` ✓
- 代码请求 -> `Action: code_executor` ✓
- 通用问题 -> `Action: direct_answer` ✓

🗣 "4/4 全部路由正确！Agent 的 Thought 部分很关键——它会先思考'这个问题需要什么工具'，然后才行动。这就是 ReAct 的精髓。"

❓ **常见问题**:
- Q: "Agent 选错工具怎么办？" -> A: "优化 system prompt 里的工具描述。描述越清晰，LLM 选择越准确"
- Q: "为什么限制 5 步？" -> A: "防止 LLM 陷入死循环。实际生产中 3-5 步足够了"


---

## 休息 + 回顾（1:00-1:10）

📍 **Cell 20**

🗣 **休息前快速回顾**:

> 到目前为止我们已经搭了三层：
> 1. **知识库** -> 文档分块 + 向量存储 + 相似度搜索
> 2. **RAG 系统** -> 检索 + LLM 生成带来源的回答
> 3. **Agent 路由** -> ReAct 循环，自动选择合适的工具
>
> 休息完我们加两个关键能力：**自动评测**和**多轮对话**。


---

## Step 4 + Checkpoint 4: LLM-as-Judge 评测（1:10-1:30）

📍 **运行 Cell 21-25**

### Cell 22: LLMJudge 类

⏱ 讲解 5 分钟

🗣 **讲课话术**:

> 系统搭好了，怎么知道它回答得好不好？让人去看？太慢了！
>
> **LLM-as-Judge**——用另一个 LLM 当"阅卷老师"。给它三样东西：
> 1. 用户问题
> 2. 系统回答
> 3. 参考标准答案
>
> 让它从三个维度打分（1-5分）：
> - **相关性**：回答是否切题
> - **准确性**：信息是否正确
> - **完整性**：是否覆盖了关键信息
>
> 平均分 >= 4.0 = 优秀，>= 3.0 = 可接受，< 3.0 = 需要优化。

### Cell 23: 评测数据集

🗣 "7 个评测样本，每个都有**参考答案**。Judge 会拿系统回答和参考答案对比。"

### Cell 25: Checkpoint 4 — 运行评测

⏱ 给 **10 分钟**（因为需要多次 LLM 调用，运行较慢）

🗣 "这一步比较慢，要调用很多次 LLM。先让它跑，我们趁这个时间聊聊评测的注意事项。"

👀 **等结果时可以聊**:
> - Judge 本身也是 LLM，也可能出错。生产中建议用更强的模型当 Judge（比如用 GPT-4 评 Qwen 的回答）
> - 评分标准要在 prompt 里写清楚，否则每次打分标准不一致
> - 可以对同一个回答评多次取平均，减少波动

👀 **结果出来后指出**:
- 各问题的三维度分数
- 平均分是否 >= 4.0
- 哪些问题得分低？为什么？

🗣 "看这个评测报告，像不像一个自动化的 QA 流程？生产中每次更新 prompt 或模型，跑一遍评测就知道有没有退化。"


---

## Step 5 + Checkpoint 5: 多轮对话（1:30-1:50）

📍 **运行 Cell 26-30**

### Cell 26-27: 为什么需要对话记忆

⏱ 讲解 5 分钟

🗣 **讲课话术**:

> 到目前为止，我们的 Agent 每次回答都是"失忆"的——不知道之前聊了什么。
>
> 举个例子：
> - 用户："StarLink 有什么产品？"
> - Agent："有 SL-3000 和 SL-3000 Pro。"
> - 用户："Pro 版有什么特别的？"
> - Agent：**"你说的 Pro 是什么？"** <-- 它忘了上一轮聊的是 StarLink！
>
> 解决方案：**ConversationMemory**——把之前的对话历史塞进 system prompt 里。

### Cell 27: ConversationMemory 类

🗣 "看代码——最多保留 5 轮对话，超出了就 FIFO（先进先出）淘汰最旧的。为什么限制 5 轮？因为 context window 有限，对话历史太长会挤占检索内容的空间。"

### Cell 28: MultiTurnAgent

🗣 "和之前的 Agent 区别就一行代码——在 system prompt 后面拼上对话历史。就这么简单！"

### Cell 30: Checkpoint 5 — 测试多轮对话

⏱ 给 **8 分钟** 动手时间

🗣 "设计一个 3-4 轮的对话序列，关键是**后面的问题要依赖前面的上下文**。"

👀 **测试 4 轮对话**:
1. "StarLink 有什么产品？" -> 列出 SL-3000 和 Pro
2. "Pro 版有什么特别的？"（引用上文） -> 正确理解"Pro"指 StarLink Pro
3. "StarView 是什么？" -> 切换话题，回答数据可视化平台
4. "买 10 台 Pro 多少钱？"（调用计算器） -> 4999 x 10 = 49990 元

🗣 "看！第 2 轮它知道 Pro 指的是 StarLink Pro，第 4 轮它先搜价格再算乘法。这就是 Agent + 记忆 + 多工具的威力！"

❓ **常见问题**:
- Q: "5 轮会不会太少？" -> A: "看你的 context window 大小。Qwen 支持 128K，理论上可以更多，但要和检索内容争空间"
- Q: "对话历史太长怎么办？" -> A: "可以做摘要压缩——用 LLM 把之前的对话总结成一段话"


---

## Step 6 + Checkpoint 6: 自由扩展 + 综合演示（1:50-2:30）

📍 **运行 Cell 31-36**

### Cell 31-34: 三个扩展选项

⏱ 给 **15 分钟** 自由发挥

🗣 **讲课话术**:

> 自由发挥时间！三个选项任选其一：
>
> **选项 A：自定义文档上传**
> - 加入你自己的文档（模拟的也行），分块、向量化、存入知识库
> - 测试能否检索到新内容
>
> **选项 B：添加自定义工具**
> - 比如日期计算："距离年底还有多少天？""今天几号？"
> - 参考上午写的 `get_datetime` 工具
>
> **选项 C：修改评测标准**
> - 加一个第 4 维度"简洁性"——回答是不是太啰嗦？
> - 修改 Judge 的 prompt
>
> 选你最感兴趣的做。做完了可以做第二个。

### Cell 35-36: 综合演示

⏱ 每人 3-5 分钟展示

🗣 "展示时间！跑这个演示脚本，5 个场景覆盖所有功能：产品查询 -> 上下文跟进 -> 价格计算 -> API 文档 -> 请假制度。"

👀 **演示要指出**:
- Agent 的 Thought 推理过程
- 工具选择是否正确
- 多轮对话上下文是否保持
- 来源引用是否准确


---

## 生产化考量（2:30-2:50）

📍 **运行 Cell 37-41**

⏱ 每个主题 5 分钟，共 20 分钟

### Cell 38: 部署方案

🗣 **讲课话术**:

> 从 Notebook 到生产，三步走：
>
> | 阶段 | 方案 | 适用场景 |
> |------|------|---------|
> | **5 分钟** | Gradio Demo | 内部演示、拿预算 |
> | **1-2 天** | FastAPI 服务 | 对接其他系统 |
> | **1-2 周** | Docker + K8s | 企业级部署 |
>
> 先用 Gradio 给老板看一眼效果，拿到预算再上 FastAPI。

### Cell 39: 监控与可观测性

🗣 "生产系统必须监控四个指标：**响应延迟**（P99 < 5s）、**LLM 调用失败率**（< 5%）、**检索命中率**（> 60%）、**用户满意度**（> 3.5/5）。"

### Cell 40: 成本优化

🗣 **讲课话术**:

> LLM API 很贵！五个省钱技巧：
> 1. **缓存**——相同问题复用答案，省 50-80%
> 2. **模型分级**——简单问题用小模型，复杂问题用大模型
> 3. **Prompt 压缩**——减少 token 数
> 4. **批量处理**——凑一批一起调用
> 5. **本地模型**——用 Ollama 部署，零 API 费用（但需要 GPU）

### Cell 41: 安全合规

🗣 "企业级系统四道防线：**输入验证**（防 Prompt 注入）、**输出过滤**（防泄露薪资/合同等敏感信息）、**权限控制**（不同角色看不同文档）、**审计日志**（记录谁问了什么）。"

❓ **常见问题**:
- Q: "Prompt 注入怎么防？" -> A: "正则过滤 + 输入前置检查 + 指令与数据分离"
- Q: "模型幻觉怎么减少？" -> A: "RAG + temperature 调低 + 要求引用来源 + 设置拒答机制"


---

## 一周行动计划（2:50-3:10）

📍 **运行 Cell 42-44**

🗣 **讲课话术**:

> 回到公司以后怎么落地？一周计划：
>
> ### Day 1：确定场景
> - 找一个高价值场景（客服知识库、内部 FAQ、产品文档问答）
> - 盘点数据资产（有哪些文档？什么格式？大概多少？）
> - 确定 LLM 方案：API（qwen-plus / GPT-4）还是本地部署（Ollama）
>
> ### Day 2-3：搭建原型
> - 复制本课程的代码搭一个 RAG 系统
> - 接入 5-10 个真实文档
> - 部署 Gradio Demo 给老板看
>
> ### Day 4-5：完善 + 优化
> - 建立评测数据集（20-50 个问答对）
> - 加多轮对话支持
> - 优化 Prompt，调整分块策略
>
> **关键提醒**：先用 Prompt + RAG 解决 80% 的问题，不要一上来就想微调！

### Cell 44: 学员写行动计划

⏱ 给 **5 分钟** 写自己的行动计划

🗣 "花 5 分钟，写下你回去要做的第一件事。写具体的——不是'研究 AI'，而是'把客服 FAQ 的 50 个问答对整理成文档，接入 RAG 系统'。"


---

## 框架对比（3:10-3:20）

📍 **运行 Cell 45-47**

🗣 **讲课话术**:

> 我们这三天全是**从零手写**的——知识库、RAG、Agent、评测。为什么不直接用框架？
>
> **因为你们现在懂原理了！** 这些框架本质上就是把我们写的东西封装了一层。

| 框架 | 定位 | 强项 | 适合场景 |
|------|------|------|---------|
| **LangChain** | Agent/Chain/RAG 全栈 | 生态最丰富 | 通用 AI 应用 |
| **LlamaIndex** | 数据框架 + RAG 专注 | RAG 管道最强 | 文档密集型系统 |
| **CrewAI** | Multi-Agent 协作 | 角色分工直观 | 复杂多步骤工作流 |
| **Dify / FastGPT** | 无代码/低代码 | 拖拽搭建 | 非技术用户 |

> 建议：先用我们的代码跑通原型，确认可行了再迁移到框架上。框架帮你省力，但出了 bug 你得能看懂底层——这就是这三天学的价值。

❓ **常见问题**:
- Q: "用 LangChain 还是 LlamaIndex？" -> A: "如果主要是文档问答，用 LlamaIndex。如果需要复杂 Agent 编排，用 LangChain"
- Q: "Dify 好用吗？" -> A: "非常适合快速原型和非技术用户。但定制性有限"


---

## 课程总结（3:20-3:30）

📍 **运行 Cell 48-50**

🗣 **三天学习总结**:

> 我们回顾一下三天都学了什么：

```
Day 1 上午: 训练循环 -> Tokenizer -> Embedding -> Prompt Engineering
Day 1 下午: Self-Attention -> Multi-Head -> Transformer Block -> GPT -> 生成策略

Day 2 上午: 预训练 Demo -> SFT 全流程 -> Loss Masking -> 超参数调优
Day 2 下午: LoRA -> QLoRA -> DPO 对齐 -> 模型评测 -> 企业成本决策

Day 3 上午: RAG 系统 -> ReAct Agent -> Code Agent -> 集成
Day 3 下午: Capstone（知识库 + RAG + Agent + 评测 + 多轮对话 + 部署）
```

> **企业决策路径**——记住这个顺序：
>
> Prompt Engineering（80% 的任务） -> RAG（需要实时知识） -> LoRA/SFT（需要高准确率） -> 全量微调（大数据 + 高要求）
>
> **最后送大家一句话**：你不是在"调 API"，你是在让 API 服务好每一个人。代码会出 bug，但你对技术的理解，才是真正的竞争力。
>
> 祝大家回去顺利落地！有问题随时联系。


---

## 附录 A：时间速查表

| Checkpoint | 时间 | 核心任务 | 预期时长 |
|-----------|------|---------|---------|
| CP1 | 0:30 | search_knowledge 函数 | 5min |
| CP2 | 0:40 | 测试 RAG 回答质量 | 5min |
| CP3 | 0:55 | 测试 Agent 路由 | 5min |
| CP4 | 1:20 | 运行 LLM-as-Judge 评测 | 10min |
| CP5 | 1:40 | 测试多轮对话 | 8min |
| CP6 | 2:15 | 综合演示 | 5min |

---

## 附录 B：关键数据速查

| 数据点 | 值 | 用途 |
|--------|------|------|
| 公司文档数 | 6 个 | 知识库规模 |
| 文档类别 | HR / 产品 / 技术 | 3 类 |
| 分块参数 | chunk_size=300, overlap=50 | 分块策略 |
| 分块后数量 | 10 个 | 向量数据库大小 |
| 搜索阈值 | 0.3 | 过滤低相关结果 |
| RAG temperature | 0.3 | 低随机性 |
| Agent 最大步数 | 5 | 防死循环 |
| 对话记忆轮数 | 最多 5 轮 | FIFO 淘汰 |
| 评测维度 | 相关性/准确性/完整性 | 1-5 分 |
| 优秀阈值 | 平均 >= 4.0 | 质量标准 |
| StarLink 基础版 | 2999 元 | SL-3000 |
| StarLink Pro | 4999 元 | SL-3000 Pro |
| 年假(1-10年) | 5 天 | HR 制度 |
| 年假(10-20年) | 10 天 | HR 制度 |
| 年假(20+年) | 15 天 | HR 制度 |
| 住宿报销上限 | 500 元/天 | 差旅 |

---

## 附录 C：应急预案

| 问题 | 解决方案 |
|------|---------|
| API 超时 | 降低 max_tokens，或换 qwen-turbo |
| Agent 选错工具 | 优化 system prompt 中的工具描述 |
| RAG 检索不准 | 调整 chunk_size 或 threshold |
| 评测太慢 | 减少评测样本数到 3-4 个 |
| 学员进度差异大 | 快的做扩展选项，慢的先完成核心 CP1-3 |
| LLM 返回格式异常 | 在 prompt 里加更严格的格式要求 |